# Milestone A — Checkpoint 3: Gold Set Awal

Notebook ini menyiapkan sampling, dua lembar anotasi independen, dan audit agreement. Notebook **tidak membuat label gold otomatis**; gold v1 baru dapat dibekukan setelah adjudikasi klinis.

In [1]:
from pathlib import Path
import json, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'Riset' / 'scripts' / 'checkpoint_03_prepare_gold.py').exists():
            return candidate
    raise FileNotFoundError('Root repository tidak ditemukan')

REPO = find_repo()
CP3 = REPO / 'Riset' / 'Tahap_2' / 'checkpoint_03'
REPO

WindowsPath('d:/Disertasi_Pipeline/cde_mapper')

## 1. Siapkan sampel dan template

Sampling bersifat deterministik dan berpasangan. Template anotasi yang sudah ada tidak ditimpa kecuali opsi overwrite diberikan secara eksplisit.

In [2]:
result = subprocess.run(
    [sys.executable, str(REPO / 'Riset/scripts/checkpoint_03_prepare_gold.py')],
    cwd=REPO, text=True, capture_output=True, check=True
)
print(result.stdout)

{
  "status": "ready_for_annotation",
  "population_pairs": 150,
  "selected_pairs": 40,
  "selected_documents": 80,
  "selected_strata": {
    "drug_or_dose": 36,
    "family_experiencer": 20,
    "general": 1,
    "long_answer": 17,
    "long_question": 17,
    "measurement": 15,
    "multi_context_question": 29,
    "negation": 39,
    "recommendation": 26,
    "temporal": 37,
    "uncertainty": 31
  },
  "template_status": {
    "annotator_a": "preserved_existing",
    "annotator_b": "preserved_existing"
  },
  "tasks_file": "Riset\\Tahap_2\\checkpoint_03\\stage2_annotation_tasks.jsonl",
  "manifest_file": "Riset\\Tahap_2\\checkpoint_03\\stage2_sampling_manifest.json"
}



In [3]:
manifest = json.loads((CP3 / 'stage2_sampling_manifest.json').read_text(encoding='utf-8'))
print('Status:', manifest['status'])
print('Populasi pasangan:', manifest['input']['pair_count'])
print('Pasangan terpilih:', manifest['sample_size'])
print('Distribusi strata:')
for key, value in manifest['selected_strata'].items():
    print(f'  {key}: {value}')

Status: ready_for_annotation
Populasi pasangan: 150
Pasangan terpilih: 40
Distribusi strata:
  drug_or_dose: 36
  family_experiencer: 20
  general: 1
  long_answer: 17
  long_question: 17
  measurement: 15
  multi_context_question: 29
  negation: 39
  recommendation: 26
  temporal: 37
  uncertainty: 31


## 2. Inspeksi contoh tugas

Periksa bahwa question dan answer dari parent record yang sama tetap berdampingan.

In [4]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

tasks = read_jsonl(CP3 / 'stage2_annotation_tasks.jsonl')
for task in tasks[:3]:
    print('\n', task['task_id'], task['strata'])
    print('Q:', task['question']['normalized_text'][:300])
    print('A:', task['answer']['normalized_text'][:300])


 gold-v1-001 ['drug_or_dose', 'family_experiencer', 'long_answer', 'multi_context_question', 'negation', 'temporal', 'uncertainty']
Q: halo dokter. suami saya lusa kemarin sehabis vaksin jenis pfizer. sebelumnya dia tidak bisa di vaksin karena punya riwayat sakit paru paru juga diabetes. tapi lusa kemarin ternyata bisa untuk mengikuti vaksin, hanya saja efek sampingnya masih terasa sampai sekarang seperti tangan bengkak, badan saki
A: alo, saat ini vaksin pfizer sudah mulai digunakan di indonesia. vaksin pfizer merupakan valsin mrna yang bekerja dengan memicu sistem sistem kekebalan tubuh membentuk spike protein, yang nantinya akan membantu tubuh membentuk antibodi yang dapat melawan virus corona.
vaksin pfizer diberikan secara i

 gold-v1-002 ['drug_or_dose', 'family_experiencer', 'multi_context_question', 'negation', 'recommendation', 'temporal', 'uncertainty']
Q: assalammualaikum dokter. selamat malam saya pria usia 33 tahun, punya riwayat infeksi saluran kemih, sekarang infeksi sa

## 3. Audit dua anotator

Sebelum anotasi selesai, status yang diharapkan adalah `pending_annotation`. Setelah A dan B lengkap serta valid, status berubah menjadi `ready_for_adjudication`, bukan langsung `pass`.

In [5]:
result = subprocess.run(
    [sys.executable, str(REPO / 'Riset/scripts/checkpoint_03_audit_annotations.py')],
    cwd=REPO, text=True, capture_output=True, check=True
)
audit = json.loads((CP3 / 'stage2_checkpoint_03_audit.json').read_text(encoding='utf-8'))
print('Status:', audit['status'])
print('Gold:', audit['gold_status'])
print('Kelengkapan:', json.dumps(audit['task_counts'], ensure_ascii=False, indent=2))
print('Agreement:', json.dumps(audit['agreement'], ensure_ascii=False, indent=2))
print('Tindakan berikutnya:', audit['next_action'])

Status: pending_annotation
Gold: not_created_pending_clinical_adjudication
Kelengkapan: {
  "expected": 40,
  "annotator_a_rows": 40,
  "annotator_b_rows": 40,
  "completed_annotator_a": 0,
  "completed_annotator_b": 0,
  "completed_by_both": 0,
  "valid_comparable_tasks": 0
}
Agreement: {
  "entity_key": [
    "start_char",
    "end_char",
    "entity_type"
  ],
  "micro_exact_span_type_f1": null,
  "macro_exact_span_type_f1": null,
  "attribute_agreement_on_matched_entities": {
    "normalized_mention": null,
    "base_entity": null,
    "domain": null,
    "value": null,
    "unit": null,
    "dose": null,
    "frequency": null,
    "route": null,
    "assertion": null,
    "temporal": null,
    "experiencer": null,
    "sentence_id": null,
    "epistemic_status": null
  },
  "disagreement_records": 0
}
Tindakan berikutnya: Isi kedua berkas anotator secara independen dan ubah status menjadi complete.


In [6]:
assert not audit['structural_errors'], audit['structural_errors']
assert audit['status'] in {'pending_annotation', 'ready_for_adjudication'}
assert 30 <= manifest['sample_size'] <= 50
print('Persiapan Checkpoint 3 lolos audit struktur. Gold v1 tetap menunggu adjudikasi klinis.')

Persiapan Checkpoint 3 lolos audit struktur. Gold v1 tetap menunggu adjudikasi klinis.
